In [30]:
import pandas as pd
from testing import TESTING_DIR
import altair as alt

from testing.r_and_d.boolean_queries.query_tester_conc import reference_df, get_question_id

BOOL_DIR = TESTING_DIR / "r_and_d/boolean_queries/outputs/"
baseline_df = (
    pd.read_json(BOOL_DIR / "baseline_conc/baseline_counts.jsonl", lines=True)
    .assign(identifier=lambda df: df["question"].apply(get_question_id))
    .assign(identifier_no=lambda df: df["identifier"].str.extract(r'(\d+)').astype(int))
)

llm_df = (
    pd.read_json(BOOL_DIR / "llm_generation/llm_counts.jsonl", lines=True)
    .assign(identifier=lambda df: df["question"].apply(get_question_id))
    .assign(identifier_no=lambda df: df["identifier"].str.extract(r'(\d+)').astype(int))
)

## BASELINE

In [36]:
fig = (
    alt.Chart(baseline_df)
    .mark_bar()
    .encode(
        x="retrieved_count:Q",
        # sort by id number
        y=alt.Y("question:N", sort=alt.EncodingSortField(
            field="identifier_no",
            order="ascending"
        )),
        tooltip = [
            alt.Tooltip("identifier:N", title="Question ID"),
            alt.Tooltip("question:N", title="Question"),
            alt.Tooltip("retrieved_count:Q", title="Retrieved count"),
            alt.Tooltip("boolean_query:N", title="Boolean query"),
    ])
    .properties(
        width=600,
        height=400,
    )
)
fig

alt.Chart(...)

In [37]:
fig = (
    alt.Chart(baseline_df)
    .mark_bar()
    .encode(
        x=alt.X("retrieved_count:Q", scale=alt.Scale(domain=[0, 100_000], clamp=True)),
        # sort by id number
        y=alt.Y("question:N", sort=alt.EncodingSortField(
            field="identifier_no",
            order="ascending"
        )),
        tooltip = [
            alt.Tooltip("identifier:N", title="Question ID"),
            alt.Tooltip("question:N", title="Question"),
            alt.Tooltip("retrieved_count:Q", title="Retrieved count"),
            alt.Tooltip("boolean_query:N", title="Boolean query"),
    ])
    .properties(
        width=600,
        height=400,
    )
)
fig

alt.Chart(...)

## LLM results

In [41]:
# get rows with errors that are null
llm_df[llm_df.model.str.contains("gpt-5")].error.iloc[0]
# llm_df.query("model == 'policy_atlas_v2'")

'Error code: 400 - {\'error\': {\'message\': "Unsupported parameter: \'max_tokens\' is not supported with this model. Use \'max_completion_tokens\' instead.", \'type\': \'invalid_request_error\', \'param\': \'max_tokens\', \'code\': \'unsupported_parameter\'}}'

In [35]:
# Filter out rows with errors (no retrieved_count)
llm_df_clean = llm_df.dropna(subset=['retrieved_count'])

# Create faceted plot: temperature vs retrieved_count by model for each question
fig = (
    alt.Chart(llm_df_clean)
    .mark_line(point=True)
    .encode(
        x=alt.X("temperature:Q", title="Temperature"),
        y=alt.Y("mean(retrieved_count):Q", title="Mean Retrieved Count"),
        color=alt.Color("model:N", title="Model"),
        facet=alt.Facet(
            "identifier:N",
            columns=3,
            title="Question",
            sort=alt.EncodingSortField(field="identifier_no", order="ascending")
        ),
        tooltip=[
            alt.Tooltip("identifier:N", title="Question ID"),
            alt.Tooltip("model:N", title="Model"),
            alt.Tooltip("temperature:Q", title="Temperature"),
            alt.Tooltip("mean(retrieved_count):Q", title="Mean Retrieved Count", format=".0f"),
        ]
    )
    .properties(
        width=200,
        height=150,
    )
    .resolve_scale(y='independent')
)
fig


alt.Chart(...)